# 04 — Entrenamiento final y submit a Kaggle

**Entrada** : `{BUCKET}/exp/<EXPERIMENTO>/resultado.json` (lo deja `03_Optuna`)
**Salida**  : `submission.csv` + submit a Kaggle

Qué hace distinto de `03`:

`03_Optuna` entrena reservando meses para validar y testear — tiene que hacerlo,
porque necesita una estimación honesta del error. Acá ya no hay nada que estimar: los
hiperparámetros están elegidos, así que se entrena con **todos los meses
supervisados**, incluidos los que `03` había apartado. Más datos, mismo modelo.

Tres cosas más:

- **Ensemble de semillas**: entrena N modelos idénticos salvo la semilla y promedia.
  Baja la varianza sin tocar el sesgo, y es de lo más barato que hay.
- **Reconstrucción a toneladas**: igual que en `03`, según la variable respuesta que
  se haya usado.
- **Armado de la entrega**: los 780 productos de `product_id_apredecir201912.txt`,
  los que no tengan predicción van con 0.

## 0 — Ambiente

In [1]:
!pip install -q uv
!uv pip install -q pyarrow polars lightgbm pandas kaggle

In [2]:
import json, os, shutil, subprocess, sys
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb


def resolver_bucket() -> Path:
    # 1) LABO3_BUCKET: para correr fuera de la nube (server propio, notebook local).
    env = os.environ.get("LABO3_BUCKET")
    if env:
        p = Path(env).expanduser().resolve()
        p.mkdir(parents=True, exist_ok=True)
        return p
    # 2) rutas conocidas: Colab y la VM de GCP
    # ~/buckets/b1 primero: es donde lo monta la instalacion de la catedra,
    # y sirve para cualquier usuario de la VM.
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1", "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Opciones: "
        "(a) Colab / VM de GCP -> corre la celda de init del ambiente; "
        "(b) server propio o local -> defini LABO3_BUCKET antes de esta celda, ej. "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET   = resolver_bucket()
RUTA_FE  = BUCKET / "datasets_fe"
RUTA_EXP = BUCKET / "exp"
RUTA_RAW = BUCKET / "datasets"

print(f"BUCKET: {BUCKET}")
print(f"\nExperimentos disponibles en {RUTA_EXP}:")
for d in sorted(RUTA_EXP.iterdir()):
    if d.is_dir() and (d / "resultado.json").exists():
        r = json.load(open(d / "resultado.json", encoding="utf-8"))
        print(f"   wape_test={r.get('wape_test', float('nan')):.4f}   {d.name}")

BUCKET: /home/ds/buckets/b1

Experimentos disponibles en /home/ds/buckets/b1/exp:
   wape_test=0.4788   grpClienteProducto_fill0_denseLife_24lags_recta_2deltas__y-norm__regression__val201907-201908_test201910-201910__cli1de4__arb500


## 1 — Palancas

`experimento` es el nombre exacto de la carpeta en `exp/`. Si lo dejás en `None`,
toma **el de menor `wape_test`** del leaderboard, que es la elección por defecto
razonable: el test es la única estimación no contaminada por la búsqueda de Optuna.

In [ ]:
PARAM = {
    # Carpeta del experimento en exp/. None = el de mejor wape_test.
    'experimento': 'grpClienteProducto_fill0_denseLife_24lags_recta_2deltas__y-delta__regression__val201907-201908_test201910-201910__cliTop50__arb500',

    # ── MUESTREO DE CLIENTES ─────────────────────────────────────────────
    # N = se queda con 1 de cada N clientes. None = todos.
    #
    # A diferencia de 03, aca NO hay val ni test que proteger: se entrena con todos los
    # meses supervisados. Pero el limite de memoria sigue existiendo: con los 1233
    # productos son 10M filas x ~610 features, y cuando LightGBM arma su matriz interna
    # eso no entra en una VM de 60 GB (medido: OOM con 62 GB de RSS).
    #
    # Con 2 entrenas con el DOBLE de datos que uso la busqueda de 03 (que va con 4),
    # que es la idea: buscar con poco, entregar con lo mas que aguante la maquina.
    #
    # Si corriste 02 con tgtFilter (780 productos) probablemente entre con None.
    # La celda de carga te imprime las filas y los GB antes de entrenar.
    'muestreo_clientes': 2,

    # ── TOP N CLIENTES POR TONELADAS ─────────────────────────────────────
    # Alternativa a 'muestreo_clientes': en vez de 1 de cada N al azar, se queda
    # con los N clientes que MAS toneladas compraron. Si esta seteada, MANDA.
    # Poné el mismo valor con que 03 busco los hiperparametros: entrenar el modelo
    # final sobre otra poblacion distinta a la de la busqueda invalida la eleccion.
    # None = desactivada.
    'top_clientes': 50,

    # ── MODO DE ENTRENAMIENTO ────────────────────────────────────────────
    # 'cluster'  -> un LightGBM por cluster (seccion 3.bis). Necesita un dataset
    #               con cluster_dtw y el resultado_clusters.json que deja 03.
    # 'ensemble' -> un unico modelo, promediando semillas (seccion 3).
    # Las celdas de la seccion que no corresponda se saltean SOLAS, asi que
    # Restart Kernel and Run All Cells funciona en los dos modos.
    'modo': 'cluster',

    # Semillas del ensemble: un modelo por semilla, se promedian las predicciones.
    # OJO CON EL COSTO: cada semilla es un entrenamiento COMPLETO. Con los
    # hiperparametros que suele elegir Optuna (500+ arboles, 200 hojas) un fit sobre
    # millones de filas x 600 features puede tardar horas.
    # La celda de calibracion de mas abajo te dice el tiempo estimado ANTES de largar.
    # [102191] = un solo modelo, y es lo razonable para la primera corrida.
    'semillas_ensemble': [102191],

    # Mes objetivo de la entrega. El pipe predice a t+horizonte; con datos hasta
    # 201912 y horizonte 2, el mes a entregar es 202002.
    'periodo_objetivo': 202002,

    # Kaggle
    'kaggle_competition': 'labo-iii-2026-rosario',
    'submit': True,                 # False = solo genera el CSV, no lo sube
    'mensaje_submit': None,         # None = se arma solo con las metricas

    # Piso de las predicciones: no existen ventas negativas.
    'clip_min': 0.0,
}
print(PARAM)

In [4]:
# ── Elegir el experimento ────────────────────────────────────────────────
disponibles = [d for d in sorted(RUTA_EXP.iterdir())
               if d.is_dir() and (d / "resultado.json").exists()]
if not disponibles:
    raise RuntimeError(f"No hay experimentos con resultado.json en {RUTA_EXP}. Corre 03_Optuna.")

if PARAM['experimento'] is None:
    def _wt(d):
        r = json.load(open(d / "resultado.json", encoding="utf-8"))
        v = r.get("wape_test")
        return float(v) if v is not None else float("inf")
    DIR_EXP = min(disponibles, key=_wt)
    print(f"Sin experimento indicado -> se elige el de mejor wape_test")
else:
    DIR_EXP = RUTA_EXP / PARAM['experimento']
    if not (DIR_EXP / "resultado.json").exists():
        raise FileNotFoundError(
            f"No existe {DIR_EXP/'resultado.json'}.\nDisponibles: {[d.name for d in disponibles]}")

CFG = json.load(open(DIR_EXP / "resultado.json", encoding="utf-8"))

EXPERIMENTO  = CFG['experimento']
DATASET_FE   = CFG['dataset_fe']
TARGET       = CFG['target']
TARGET_KIND  = CFG['target_kind']
METODO       = CFG['metodo_normalizacion']
H            = CFG['horizonte']
FEATURES     = CFG['features']
CAT_FEATURES = CFG['cat_features']
HIPER        = CFG['hiperparametros']

print(f"Experimento : {EXPERIMENTO}")
print(f"Dataset FE  : {DATASET_FE}")
print(f"Target      : {TARGET}  (kind={TARGET_KIND}, norm={METODO}, horizonte={H})")
print(f"Features    : {len(FEATURES)}   categoricas: {CAT_FEATURES}")
print(f"WAPE val    : {CFG.get('wape_val')}")
print(f"WAPE test   : {CFG.get('wape_test')}")
print(f"Leakage     : {CFG.get('leakage')}")
print(f"\nHiperparametros:")
for k, v in HIPER.items():
    print(f"   {k:22s} {v}")

Sin experimento indicado -> se elige el de mejor wape_test
Experimento : grpClienteProducto_fill0_denseLife_24lags_recta_2deltas__y-norm__regression__val201907-201908_test201910-201910__cli1de4__arb500
Dataset FE  : preprocesado_grpClienteProducto_fill0_denseLife_24lags_recta_2deltas.parquet
Target      : clase_tn_norm  (kind=norm, norm=recta, horizonte=2)
Features    : 609   categoricas: ['cat1', 'cat2', 'cat3', 'brand']
WAPE val    : 0.19005049421474887
WAPE test   : 0.4787578685816245
Leakage     : 0 error(es), 0 warning(s)

Hiperparametros:
   num_leaves             92
   max_depth              8
   learning_rate          0.019023357454464613
   n_estimators           421
   min_child_samples      100
   subsample              0.5712320240607551
   colsample_bytree       0.671179408803904
   reg_alpha              1.3818202190605103e-05
   reg_lambda             2.178290091729982e-07


## 2 — Datos

In [ ]:
path_in = RUTA_FE / DATASET_FE
if not path_in.exists():
    raise FileNotFoundError(f"No existe {path_in}. Corre 02_FE con esas palancas.")

# ══ CARGA: mismo tratamiento de memoria que 03 ══════════════════════════════
# 1) Float32 en las features (LightGBM las binariza en 255 bins: la precision extra
#    de Float64 se tira igual). Se exceptuan las columnas de la aritmetica de
#    toneladas, donde el round-trip exige error < 1e-6.
# 2) Muestreo de clientes por hash (PARAM['muestreo_clientes']), deterministico y
#    conservando la historia completa de cada cliente elegido.
# 3) scan_parquet + collect ya filtrado: el dataframe completo nunca se materializa.
CTX_F64 = {'B0', 'B1', 'tn0_norm', 'tn0',
           'clase_tn', 'clase_tn_norm', 'clase_tn_delta'}

lf = pl.scan_parquet(path_in)
_schema = lf.collect_schema()
COLS_ALL = list(_schema.keys())

_f64 = [c for c, t in _schema.items() if t == pl.Float64]
_a_f32 = [c for c in _f64 if c not in CTX_F64]
lf = lf.with_columns([pl.col(c).cast(pl.Float32) for c in _a_f32])

periodos = sorted(lf.select('periodo').unique().collect()['periodo'].to_list())
print(f"Dataset: {len(COLS_ALL)} columnas   periodos {periodos[0]} -> {periodos[-1]}")
print(f"Float64 -> Float32: {len(_a_f32)} de {len(_f64)} columnas")

# ── Inferencia: target nulo Y en los ultimos H meses ────────────────────────
# Igual que en 03: los nulos son de dos clases. Los de los ultimos H meses son lo que
# hay que predecir; el resto es fin de vida del par producto-cliente (denseLife) y no
# sirve para nada. Sin este filtro, infer_pd trae filas historicas de todos los meses.
MESES_INFER = periodos[-H:]

_N_CLI   = PARAM.get('muestreo_clientes')
_TOP_CLI = PARAM.get('top_clientes')
_hay_cli = 'customer_id' in COLS_ALL

if _TOP_CLI and _hay_cli:
    # ── TOP N CLIENTES POR TONELADAS ────────────────────────────────────────
    # Mismo criterio que 03: el ranking sale de los meses supervisados, que son los
    # que se usan para entrenar. Los de inferencia no participan de la seleccion.
    _col_tn = next((c for c in ('tn0', 'tn') if c in COLS_ALL), None)
    if _col_tn is None:
        raise RuntimeError("No encuentro columna de toneladas ('tn0' ni 'tn') "
                           "para rankear clientes.")
    _rk = (lf.filter(pl.col(TARGET).is_not_null())
             .group_by('customer_id')
             .agg(pl.col(_col_tn).sum().alias('_tn'))
             .sort('_tn', descending=True)
             .head(int(_TOP_CLI))
             .collect())
    _cli_ok = pl.col('customer_id').is_in(_rk['customer_id'].to_list())
    print(f"Top {_rk.height} clientes por {_col_tn}"
          + (f"   ('muestreo_clientes'={_N_CLI} ignorado)" if _N_CLI else ""))
    if CFG.get('top_clientes') and CFG['top_clientes'] != _TOP_CLI:
        print(f"   [AVISO] 03 busco los hiperparametros con top {CFG['top_clientes']}, "
              f"aca estas usando top {_TOP_CLI}. Son poblaciones distintas.")
elif _N_CLI and _hay_cli:
    _cli_ok = pl.col('customer_id').hash(seed=CFG.get('semilla', 102191)) % int(_N_CLI) == 0
else:
    _cli_ok = pl.lit(True)

df_infer = lf.filter(pl.col(TARGET).is_null()
                     & pl.col('periodo').is_in(MESES_INFER)).collect()
df_sup = lf.filter(pl.col(TARGET).is_not_null() & _cli_ok).collect()

_sup_disponibles = lf.select(pl.col(TARGET).is_not_null().sum()).collect().item()
meses_sup = sorted(df_sup['periodo'].unique().to_list())
meses_inf = sorted(df_infer['periodo'].unique().to_list())

print(f"\nSupervisado : {df_sup.height:,} de {_sup_disponibles:,} filas "
      f"({100 * df_sup.height / _sup_disponibles:.0f}%)"
      + (f"   [1 de cada {_N_CLI} clientes]" if _N_CLI else "   [todos los clientes]"))
print(f"              meses {meses_sup[0]} -> {meses_sup[-1]}")
print(f"Inferencia  : {df_infer.height:,} filas   meses {meses_inf}")
print(f"\nRAM del split: {(df_sup.estimated_size() + df_infer.estimated_size()) / 1e9:.2f} GB")

# OJO: la inferencia NO se muestrea. Se predicen todas las filas de los ultimos H
# meses, porque de ahi sale la entrega y no puede faltar ningun producto.

# A diferencia de 03, ACA SE ENTRENA CON TODO lo supervisado: los meses que 03
# habia reservado para val y test ya cumplieron su funcion.
extra = sorted(set(meses_sup) - set(CFG['meses_train']))
print(f"\n03 entreno con {len(CFG['meses_train'])} meses; aca se usan {len(meses_sup)}.")
print(f"Meses que 03 no habia usado para entrenar: {extra}")
# Con que poblacion de clientes busco 03 los hiperparametros. top_clientes manda
# sobre muestreo_clientes, igual que en 03: si no, el mensaje diria una cosa y se
# habria usado otra. n_filas_train puede no estar (resultado.json por cluster).
_pob = (f"top {CFG['top_clientes']} clientes" if CFG.get('top_clientes')
        else (f"1 de cada {CFG['muestreo_clientes']} clientes"
              if CFG.get('muestreo_clientes') else None))
if _pob:
    _nf = CFG.get('n_filas_train')
    print(f"03 busco con {_pob}" + (f" ({_nf:,} filas de train)." if _nf else "."))

In [ ]:
import gc

IDS = [c for c in ['product_id', 'customer_id', 'Agrupacion_ID'] if c in COLS_ALL]
# clase_tn entra al contexto para poder validar el round-trip sin necesitar df_sup
# (que se libera al final de esta celda).
# cluster_dtw NO es una feature: es la clave de particion que usa la seccion 3.bis
# para saber que modelo aplica a cada fila. Va en el contexto para que viaje igual.
COLS_CTX = [c for c in ['B0', 'B1', 'tn0_norm', 'clase_tn', 'cluster_dtw'] + IDS
            if c in COLS_ALL]

faltan = [c for c in FEATURES if c not in COLS_ALL]
if faltan:
    raise ValueError(f"El dataset no tiene {len(faltan)} features del experimento: {faltan[:10]}")

_cols = sorted(set(FEATURES + COLS_CTX + ['periodo'] + [TARGET]))
_cats = [c for c in CAT_FEATURES if c in _cols]

# Las categoricas se castean en POLARS antes de pasar a pandas: asi llegan como
# 'category' y no como object dtype (un str de Python por celda, GB de basura).
train_pd = (df_sup.select(_cols)
                  .with_columns([pl.col(c).cast(pl.Categorical) for c in _cats])
                  .to_pandas())
infer_pd = (df_infer.select([c for c in _cols if c in df_infer.columns])
                    .with_columns([pl.col(c).cast(pl.Categorical)
                                   for c in _cats if c in df_infer.columns])
                    .to_pandas())

# Las categoricas de inferencia deben compartir el mismo diccionario que las de train
for c in CAT_FEATURES:
    train_pd[c] = train_pd[c].astype('category')
    if c in infer_pd.columns:
        infer_pd[c] = infer_pd[c].astype('category').cat.set_categories(
            train_pd[c].cat.categories)

print(f"train: {train_pd.shape}   inferencia: {infer_pd.shape}")
if infer_pd.empty:
    raise RuntimeError("No hay filas de inferencia: sin ellas no se puede armar la entrega.")

# ── Liberar polars ──────────────────────────────────────────────────────────
# De aca en adelante todo trabaja sobre train_pd / infer_pd. Dejar df_sup y df_infer
# vivos duplicaria el dataset en RAM justo cuando LightGBM necesita el espacio.
for _v in ('df', 'df_sup', 'df_infer'):
    globals().pop(_v, None)
gc.collect()
print(f"polars liberado; train_pd en RAM: "
      f"{train_pd.memory_usage(deep=True).sum() / 1e9:.2f} GB"
      f"   infer_pd: {infer_pd.memory_usage(deep=True).sum() / 1e9:.2f} GB")

## 3 — Entrenamiento del ensemble

Cada semilla cambia el submuestreo de filas y de columnas, así que los modelos
cometen errores distintos. Promediarlos cancela parte de esa varianza.

`deterministic=True` para que dos corridas del notebook den el mismo submit — sin
eso, LightGBM multihilo no es reproducible y no se puede rastrear qué generó cada
entrega.

> **Antes de largar, la celda de abajo estima cuánto va a tardar.** Cada semilla es un
> entrenamiento completo, y Optuna tiende a elegir modelos grandes: con 1366 árboles y
> 189 hojas sobre 300k filas, un solo fit puede llevar 40 minutos. Mirá el estimado y
> recién después decidí cuántas semillas te conviene poner.


In [ ]:
if PARAM.get('modo') == 'cluster':
    print('Modo cluster: se saltea el ensemble de la seccion 3.')
else:
    def params_finales(semilla):
        p = dict(HIPER)
        p.update({
            'objective':      CFG['objective_lgbm'],
            'metric':         'mae',
            'verbosity':      -1,
            'boosting_type':  'gbdt',
            'n_jobs':         -1,
            'subsample_freq': 1,
            'seed':           semilla,
            'deterministic':  True,
            'force_row_wise': True,
        })
        return p


    # ── La matriz de entrenamiento, UNA sola vez ─────────────────────────────────
    # train_pd[FEATURES] hace una copia completa. Antes se llamaba una vez por semilla
    # (mas una para la calibracion), asi que con N semillas se alocaban N+1 copias de
    # varios GB cada una. Extrayendola una vez y reusandola, se paga una sola.
    X = train_pd[FEATURES]
    y = train_pd[TARGET]
    print(f"Matriz de entrenamiento: {X.shape[0]:,} filas x {X.shape[1]} features"
          f"   ({X.memory_usage(deep=True).sum() / 1e9:.2f} GB)")

    # ── Cuanto va a tardar esto ──────────────────────────────────────────────
    # Se entrena un modelo identico pero con pocos arboles y se extrapola. El tiempo de
    # LightGBM crece casi lineal en n_estimators, asi que la cuenta es confiable.
    import time

    N_CALIB = 40
    _p = params_finales(PARAM['semillas_ensemble'][0])
    _n_real = int(_p.get('n_estimators', 100))
    _p_calib = dict(_p); _p_calib['n_estimators'] = N_CALIB

    _t0 = time.time()
    _m = lgb.LGBMRegressor(**_p_calib)
    _m.fit(X, y, categorical_feature=CAT_FEATURES)
    _t_calib = time.time() - _t0
    del _m
    gc.collect()

    _por_modelo = _t_calib * _n_real / N_CALIB
    _total = _por_modelo * len(PARAM['semillas_ensemble'])

    def _fmt(seg):
        if seg < 90:  return f"{seg:.0f} s"
        if seg < 5400: return f"{seg/60:.1f} min"
        return f"{seg/3600:.1f} h"

    print(f"\nHiperparametros elegidos: {_n_real} arboles, {_p.get('num_leaves')} hojas")
    print(f"Datos: {len(X):,} filas x {len(FEATURES)} features")
    print(f"Calibracion con {N_CALIB} arboles: {_t_calib:.1f} s")
    print()
    print(f"ESTIMADO  por modelo : {_fmt(_por_modelo)}")
    print(f"          {len(PARAM['semillas_ensemble'])} semilla(s) : {_fmt(_total)}")
    print()
    if _total > 3600:
        print("Mas de una hora. Si estas en una maquina spot, considera:")
        print("  - bajar 'semillas_ensemble' a una sola")
        print("  - subir 'muestreo_clientes' (entrena con menos filas)")
        print("  - elegir un experimento con 'techo_arboles' mas bajo")


    modelos = []
    for i, sem in enumerate(PARAM['semillas_ensemble'], 1):
        m = lgb.LGBMRegressor(**params_finales(sem))
        m.fit(X, y, categorical_feature=CAT_FEATURES)
        modelos.append(m)
        print(f"[{i}/{len(PARAM['semillas_ensemble'])}] semilla {sem} entrenada")

    _filas_train = len(X)
    del X, y
    gc.collect()

    print(f"\n{len(modelos)} modelo(s) en el ensemble, {_filas_train:,} filas cada uno")

## 3.bis — Entrenamiento final por cluster

Reemplaza al entrenamiento único de la sección 3. Entrena un LightGBM por cluster, cada
uno con los hiperparámetros que le encontró `03_Optuna_dtw`, y arma un único vector de
predicciones concatenando las de cada modelo.

Los clusters que `03` salteó por chicos, y las filas con `cluster_dtw = -1` (productos
sin serie suficiente para clusterizar), caen en un **modelo de respaldo** entrenado con
todo el dataset. Sin eso te quedarías sin predicción para esos productos, y Kaggle
espera los 780.

In [ ]:
if PARAM.get('modo') != 'cluster':
    print('Modo ensemble: se saltea la seccion 3.bis (por cluster).')
else:
    # ══ UN MODELO POR CLUSTER ══════════════════════════════════════════════════
    _path_cl = DIR_EXP / 'resultado_clusters.json'
    if not _path_cl.exists():
        raise FileNotFoundError(
            f"No existe {_path_cl}. Corre la seccion 6.bis de 03_Optuna_dtw, "
            "que es la que lo escribe.")

    CFG_CL   = json.load(open(_path_cl, encoding='utf-8'))
    POR_CL   = {int(k): v for k, v in CFG_CL['por_cluster'].items()}
    print(f"Clusters con hiperparametros propios: {sorted(POR_CL)}")

    if 'cluster_dtw' not in train_pd.columns:
        raise RuntimeError("El dataset no tiene cluster_dtw. Usa el generado por 02_FE_dtw.")

    modelos_cluster, cobertura = {}, {}

    # ── Un modelo por cluster ──────────────────────────────────────────────────
    for k, info in sorted(POR_CL.items()):
        _m = train_pd['cluster_dtw'] == k
        print(f"\ncluster {k}: {_m.sum():,} filas de entrenamiento")
        _p = dict(info['hiperparametros'])
        _p.update({
            'objective':     CFG['objective_lgbm'],
            'metric':        'mae',
            'verbosity':     -1,
            'boosting_type': 'gbdt',
            'n_jobs':        -1,
            'seed':          PARAM['semillas_ensemble'][0],
            'deterministic': True,
        })
        _n = _p.pop('n_estimators', 500)
        modelos_cluster[k] = lgb.train(
            _p,
            lgb.Dataset(train_pd.loc[_m, FEATURES], label=train_pd.loc[_m, TARGET],
                        categorical_feature=CAT_FEATURES, free_raw_data=True),
            num_boost_round=_n,
        )
        cobertura[k] = int(_m.sum())

    # ── Modelo de respaldo, para clusters sin hiperparametros propios ──────────
    # Cubre el cluster -1 y los que 03 salteo por tener pocas filas.
    _huerfanos = sorted(set(int(c) for c in train_pd['cluster_dtw'].unique()) - set(POR_CL))
    if _huerfanos:
        print(f"\nClusters sin modelo propio: {_huerfanos} -> se entrena un modelo de respaldo")
        _p = dict(HIPER)
        _p.update({
            'objective':     CFG['objective_lgbm'],
            'metric':        'mae',
            'verbosity':     -1,
            'boosting_type': 'gbdt',
            'n_jobs':        -1,
            'seed':          PARAM['semillas_ensemble'][0],
            'deterministic': True,
        })
        _n = _p.pop('n_estimators', 500)
        modelos_cluster['respaldo'] = lgb.train(
            _p,
            lgb.Dataset(train_pd[FEATURES], label=train_pd[TARGET],
                        categorical_feature=CAT_FEATURES, free_raw_data=True),
            num_boost_round=_n,
        )

    print(f"\nModelos entrenados: {list(modelos_cluster)}")


In [ ]:
if PARAM.get('modo') != 'cluster':
    print('Modo ensemble: se saltea la seccion 3.bis (por cluster).')
else:
    # ── Prediccion: cada fila la predice el modelo de SU cluster ───────────────
    _pred = np.full(len(infer_pd), np.nan, dtype=np.float64)
    _cl_infer = infer_pd['cluster_dtw'].to_numpy()

    for k in sorted(set(int(c) for c in np.unique(_cl_infer))):
        _m = _cl_infer == k
        _modelo = modelos_cluster.get(k, modelos_cluster.get('respaldo'))
        if _modelo is None:
            print(f"[aviso] cluster {k}: {_m.sum():,} filas SIN modelo, quedan en NaN")
            continue
        _pred[_m] = _modelo.predict(infer_pd.loc[_m, FEATURES])
        _cual = k if k in modelos_cluster else 'respaldo'
        print(f"cluster {k}: {_m.sum():,} filas predichas con el modelo {_cual}")

    _faltan = int(np.isnan(_pred).sum())
    print(f"\nFilas sin prediccion: {_faltan:,}")
    if _faltan:
        print("   ^ revisalas antes de submitear: van a faltar productos en la entrega")

    # La seccion 4 detecta esta variable y usa estas predicciones en lugar de las del
    # ensemble. Esta en la escala del TARGET; alli se reconstruye a toneladas.
    pred_target_cluster = _pred


## 4 — Predicción y reconstrucción a toneladas

Sea cual sea la variable respuesta, la predicción del modelo se lleva a toneladas
con los `B0`/`B1` de cada fila. Es el inverso exacto de la normalización de `02_FE`.

In [ ]:
def reconstruir_nivel(pred, ctx: pd.DataFrame) -> np.ndarray:
    """Pasa la prediccion a toneladas segun la variable respuesta del experimento."""
    pred = np.asarray(pred, dtype=np.float64)
    if TARGET_KIND == 'nivel':
        return pred
    if TARGET_KIND == 'delta':
        # clase_tn_delta = clase_tn_norm - tn0_norm
        pred = pred + ctx['tn0_norm'].to_numpy(dtype=np.float64)
    B0 = ctx['B0'].to_numpy(dtype=np.float64)
    B1 = ctx['B1'].to_numpy(dtype=np.float64)
    if METODO == 'recta':
        return pred + (B0 + B1 * (-float(H)))
    B1s = np.where((B1 == 0) | ~np.isfinite(B1), 1.0, B1)
    return pred * B1s + B0


# Chequeo de sanidad: reconstruir el TARGET REAL sobre train debe devolver clase_tn.
# clase_tn viaja dentro de train_pd (esta en COLS_CTX), asi que no hace falta df_sup
# -- que ya fue liberado -- ni asumir que los dos mantienen el mismo orden de filas.
if 'clase_tn' in train_pd.columns:
    _m = train_pd.head(20_000)
    _rec = reconstruir_nivel(_m[TARGET].to_numpy(), _m)
    _err = float(np.nanmax(np.abs(_rec - _m['clase_tn'].to_numpy())))
    print(f"Round-trip de reconstruccion: error maximo = {_err:.10f}")
    if _err > 1e-6:
        raise RuntimeError(f"La reconstruccion no cierra (error {_err}). Revisa METODO={METODO!r}.")
    print("Reconstruccion validada.")

# Promedio del ensemble sobre la escala del target, y despues a toneladas
# Dos caminos: si corriste la seccion 3.bis (un modelo por cluster) se usan esas
# predicciones; si corriste la seccion 3 (ensemble de semillas), se promedian esas.
if 'pred_target_cluster' in globals():
    print("Usando las predicciones POR CLUSTER (seccion 3.bis)")
    y_pred = reconstruir_nivel(pred_target_cluster, infer_pd)
else:
    print(f"Usando el ensemble de {len(modelos)} semillas (seccion 3)")
    preds = np.column_stack([m.predict(infer_pd[FEATURES]) for m in modelos])
    y_pred = reconstruir_nivel(preds.mean(axis=1), infer_pd)
y_pred = np.maximum(y_pred, PARAM['clip_min'])

def sumar_meses(p, k):
    m = (p // 100) * 12 + (p % 100) - 1 + k
    return (m // 12) * 100 + (m % 12) + 1

pred = infer_pd[['periodo'] + IDS].copy()
pred['tn_pred'] = y_pred
pred['periodo_objetivo'] = pred['periodo'].map(lambda p: sumar_meses(int(p), H))

print(f"\nPredicciones: {len(pred):,} filas")
print(pred.groupby(['periodo', 'periodo_objetivo']).size().rename('filas').reset_index().to_string(index=False))
print(f"\ntn_pred   min {y_pred.min():.3f}   media {y_pred.mean():.3f}   max {y_pred.max():.3f}")
print(f"predicciones en 0: {(y_pred == 0).sum():,}")

## 5 — Armado de la entrega

Kaggle mide a nivel `product_id`. Con granularidad producto-cliente hay que **sumar
las predicciones de todos los clientes** de cada producto — es la misma agregación
con la que `03` midió el WAPE, así que el número de la entrega es comparable con el
del leaderboard.

Los productos de la lista oficial que no tengan predicción van con **0**. Que sean
muchos es una señal de alarma, no algo normal: significa que el pipe perdió
productos en el camino.

In [9]:
OBJ = PARAM['periodo_objetivo']
pred_obj = pred[pred['periodo_objetivo'] == OBJ]
if pred_obj.empty:
    raise RuntimeError(
        f"No hay predicciones para {OBJ}. Objetivos disponibles: "
        f"{sorted(pred['periodo_objetivo'].unique())}")

por_producto = (pred_obj.groupby('product_id', as_index=False)['tn_pred']
                        .sum().rename(columns={'tn_pred': 'tn'}))
print(f"Mes objetivo {OBJ}: {len(pred_obj):,} filas -> {len(por_producto)} productos")

path_apredecir = RUTA_RAW / "product_id_apredecir201912.txt"
if not path_apredecir.exists():
    raise FileNotFoundError(f"Falta {path_apredecir}")
oficiales = pl.read_csv(path_apredecir, separator="\t").to_pandas()
print(f"Productos en la lista oficial: {len(oficiales)}")

submit = (oficiales[['product_id']]
          .merge(por_producto, on='product_id', how='left'))
sin_pred = submit['tn'].isna().sum()
submit['tn'] = submit['tn'].fillna(0.0)
submit = submit.sort_values('product_id').reset_index(drop=True)

print(f"\nSubmit: {len(submit)} filas")
print(f"Productos SIN prediccion (van en 0): {sin_pred}")
if sin_pred > 0:
    faltantes = submit.loc[submit['tn'] == 0, 'product_id'].tolist()
    print(f"   {faltantes[:20]}{' ...' if len(faltantes) > 20 else ''}")
    if sin_pred > len(oficiales) * 0.05:
        print(f"   ATENCION: es mas del 5% de la lista. Revisa si 01_Preprocesamiento")
        print(f"   corrio con filter_target_products_only=True y sin muestreo.")
print(f"\ntn   min {submit['tn'].min():.3f}   media {submit['tn'].mean():.3f}   "
      f"max {submit['tn'].max():.3f}   suma {submit['tn'].sum():,.1f}")
print(submit.head(10).to_string(index=False))

Mes objetivo 202002: 297,019 filas -> 927 productos
Productos en la lista oficial: 780

Submit: 780 filas
Productos SIN prediccion (van en 0): 0

tn   min 0.066   media 35.000   max 1397.428   suma 27,300.0
 product_id          tn
      20001 1397.428085
      20002 1122.964675
      20003  714.145431
      20004  497.617352
      20005  519.015576
      20006  405.070619
      20007  405.151617
      20008  360.278627
      20009  500.237412
      20010  345.615929


In [10]:
path_submit = DIR_EXP / f"submission_{OBJ}.csv"
submit.to_csv(path_submit, index=False)
print(f"Guardado: {path_submit}")

# Copia con nombre fijo, para encontrar siempre la ultima
shutil.copy(path_submit, RUTA_EXP / "submission_ultima.csv")
print(f"Copia    : {RUTA_EXP/'submission_ultima.csv'}")
print()
print(open(path_submit).read()[:300])

Guardado: /home/ds/buckets/b1/exp/grpClienteProducto_fill0_denseLife_24lags_recta_2deltas__y-norm__regression__val201907-201908_test201910-201910__cli1de4__arb500/submission_202002.csv
Copia    : /home/ds/buckets/b1/exp/submission_ultima.csv

product_id,tn
20001,1397.4280850414798
20002,1122.9646753563625
20003,714.145430988946
20004,497.6173518049602
20005,519.0155756258556
20006,405.0706188646754
20007,405.15161686046036
20008,360.278626695125
20009,500.23741229158503
20010,345.61592863598275
20011,329.08915360659773
20012,294.14324713


## 6 — Submit a Kaggle

Necesita `~/.kaggle/kaggle.json` con permisos `600`. La celda lo busca en el home y,
si no está, lo copia del bucket.

In [11]:
kaggle_dst = Path.home() / ".kaggle" / "kaggle.json"
kaggle_dst.parent.mkdir(parents=True, exist_ok=True)

if kaggle_dst.exists():
    kaggle_dst.chmod(0o600)
    print(f"Kaggle auth OK: {kaggle_dst}")
else:
    for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
        if cand.exists():
            shutil.copy(cand, kaggle_dst)
            kaggle_dst.chmod(0o600)
            print(f"Kaggle auth copiada de {cand}")
            break
    else:
        print("kaggle.json NO encontrado.")
        print("Bajalo de kaggle.com -> Settings -> API -> Create New Token")
        print(f"y dejalo en {kaggle_dst} o en {BUCKET}/kaggle.json")

Kaggle auth OK: /home/ds/.kaggle/kaggle.json


In [12]:
def kaggle_cli(args):
    """Corre la CLI de kaggle. Devuelve (ok, salida). Nunca lanza excepcion:
    el CSV ya esta generado y no vale la pena romper la corrida por el submit."""
    try:
        r = subprocess.run(['kaggle'] + args, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout or '') + (r.stderr or '')
    except FileNotFoundError:
        return False, ("La CLI de kaggle no esta instalada o no esta en el PATH.\n"
                       "   Instalala con:  pip install kaggle\n"
                       "   El CSV ya quedo generado; podes subirlo a mano desde la web.")
    except Exception as e:
        return False, f"Error inesperado llamando a kaggle: {type(e).__name__}: {e}"


if not PARAM['submit']:
    print("PARAM['submit'] = False -> no se sube nada. El CSV ya esta generado.")
elif not kaggle_dst.exists():
    print("Sin credenciales de Kaggle: no se sube. El CSV ya esta generado.")
else:
    msg = PARAM['mensaje_submit'] or (
        f"{EXPERIMENTO[:80]} | wape_test={CFG.get('wape_test')} | "
        f"ensemble={len(modelos)} semillas")
    ok, salida = kaggle_cli(['competitions', 'submit',
                             '-c', PARAM['kaggle_competition'],
                             '-f', str(path_submit),
                             '-m', msg])
    print(f"mensaje: {msg}")
    print(salida)
    print("Submit enviado. Verificalo con la celda de abajo." if ok
          else "NO se pudo subir. El CSV esta en disco, se puede subir a mano.")

mensaje: grpClienteProducto_fill0_denseLife_24lags_recta_2deltas__y-norm__regression__val | wape_test=0.4787578685816245 | ensemble=1 semillas
99 submissions remaining today.
Successfully submitted to Labo III, 2026 Rosario
  0%|          | 0.00/18.6k [00:00<?, ?B/s]
100%|██████████| 18.6k/18.6k [00:00<00:00, 58.6kB/s]

Submit enviado. Verificalo con la celda de abajo.


In [13]:
# Ultimos submits de la competencia
ok, salida = kaggle_cli(['competitions', 'submissions', '-c', PARAM['kaggle_competition']])
print(salida if salida.strip() else "(sin respuesta)")


     ref  fileName                          date                        description                                                                                                                            status                     publicScore  privateScore  
--------  --------------------------------  --------------------------  -------------------------------------------------------------------------------------------------------------------------------------  -------------------------  -----------  ------------  
55125271  submission_202002.csv             2026-07-31 02:58:51.977000  grpClienteProducto_fill0_denseLife_24lags_recta_2deltas__y-norm__regression__val | wape_test=0.4787578685816245 | ensemble=1 semillas  SubmissionStatus.PENDING                              
53584327  tb_kaggle_muestra.csv             2026-06-12 00:47:46.597000  prueba de kaggle verificar                                                                                                             Submiss

## 7 — Registro de la entrega

Deja constancia de qué modelo generó qué submit, para poder reconstruirlo después.

In [14]:
registro = {
    'experimento':        EXPERIMENTO,
    'dataset_fe':         DATASET_FE,
    'periodo_objetivo':   OBJ,
    'archivo':            str(path_submit),
    'n_productos':        int(len(submit)),
    'n_sin_prediccion':   int(sin_pred),
    'tn_total_predicho':  float(submit['tn'].sum()),
    'semillas_ensemble':  PARAM['semillas_ensemble'],
    'meses_entrenamiento': [int(m) for m in meses_sup],
    # Con cuantos datos se entrego, y con cuantos se habian buscado los hiperparametros
    # en 03. Sin esto no se puede explicar despues por que dos submits del mismo
    # experimento dieron distinto.
    'muestreo_clientes':      PARAM.get('muestreo_clientes'),
    'n_filas_entrenamiento':  int(_filas_train),
    'muestreo_clientes_en_03': CFG.get('muestreo_clientes'),
    'n_filas_train_en_03':     CFG.get('n_filas_train'),
    'wape_val_en_03':     CFG.get('wape_val'),
    'wape_test_en_03':    CFG.get('wape_test'),
    'hiperparametros':    HIPER,
}
with open(DIR_EXP / f"submit_{OBJ}.json", 'w', encoding='utf-8') as f:
    json.dump(registro, f, indent=2, ensure_ascii=False, default=str)

print(f"Registro: {DIR_EXP/f'submit_{OBJ}.json'}")
print(json.dumps({k: v for k, v in registro.items() if k != 'hiperparametros'},
                 indent=2, ensure_ascii=False, default=str))

Registro: /home/ds/buckets/b1/exp/grpClienteProducto_fill0_denseLife_24lags_recta_2deltas__y-norm__regression__val201907-201908_test201910-201910__cli1de4__arb500/submit_202002.json
{
  "experimento": "grpClienteProducto_fill0_denseLife_24lags_recta_2deltas__y-norm__regression__val201907-201908_test201910-201910__cli1de4__arb500",
  "dataset_fe": "preprocesado_grpClienteProducto_fill0_denseLife_24lags_recta_2deltas.parquet",
  "periodo_objetivo": 202002,
  "archivo": "/home/ds/buckets/b1/exp/grpClienteProducto_fill0_denseLife_24lags_recta_2deltas__y-norm__regression__val201907-201908_test201910-201910__cli1de4__arb500/submission_202002.csv",
  "n_productos": 780,
  "n_sin_prediccion": 0,
  "tn_total_predicho": 27300.018307494207,
  "semillas_ensemble": [
    102191
  ],
  "meses_entrenamiento": [
    201701,
    201702,
    201703,
    201704,
    201705,
    201706,
    201707,
    201708,
    201709,
    201710,
    201711,
    201712,
    201801,
    201802,
    201803,
    201804,
